# Pré-processamento — Detecção de Fraude em Transações Financeiras

**Projeto:** Detecção de fraude em transações financeiras com PaySim  
**Entregável:** Notebook de pré-processamento  
**Base:** decisões tomadas no notebook de EDA refatorado

## Objetivo deste notebook

Este notebook transforma as decisões da EDA em uma base pronta para modelagem.

A ideia é separar claramente as responsabilidades:

- **EDA:** entender os dados e justificar decisões;
- **Pré-processamento:** aplicar transformações de forma reprodutível;
- **Modelagem:** treinar modelos, aplicar estratégias de desbalanceamento e comparar métricas.

Neste notebook **não serão treinados modelos** e **não será aplicado SMOTE**, pois técnicas de balanceamento devem ser testadas somente no conjunto de treino durante a modelagem.


## 1. Importações e configurações

A configuração abaixo define:

- caminhos possíveis para encontrar o CSV bruto do PaySim;
- pasta de saída dos arquivos processados;
- tolerância usada para inconsistências de saldo;
- semente aleatória para reprodutibilidade.


In [18]:
from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd
import kagglehub

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TOLERANCIA_SALDO = 1e-2

KAGGLE_DATASET = "ealaxi/paysim1"

OUTPUT_DIR = Path("data/processado")

TIPOS_TRANSACAO_ESPERADOS = [
    "PAYMENT",
    "TRANSFER",
    "CASH_OUT",
    "DEBIT",
    "CASH_IN",
]

COLUNAS_ESPERADAS = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud",
    "isFlaggedFraud",
]

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 2. Carregamento da base bruta


In [19]:
caminho_dataset = Path(kagglehub.dataset_download(KAGGLE_DATASET))

print(f"Dataset disponível em: {caminho_dataset}")

arquivos_csv = list(caminho_dataset.rglob("*.csv"))

if len(arquivos_csv) == 0:
    raise FileNotFoundError(
        "Nenhum arquivo CSV foi encontrado dentro da pasta baixada pelo KaggleHub."
    )

dataset_path = arquivos_csv[0]

print(f"Arquivo CSV encontrado: {dataset_path}")

df = pd.read_csv(dataset_path)

print(f"Dimensão original: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas")

df.head()

Using Colab cache for faster access to the 'paysim1' dataset.
Dataset disponível em: /kaggle/input/paysim1
Arquivo CSV encontrado: /kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv
Dimensão original: 6,362,620 linhas x 11 colunas


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,"9,839.640000",C1231006815,"170,136.000000","160,296.360000",M1979787155,0.000000,0.000000,0,0
1,1,PAYMENT,"1,864.280000",C1666544295,"21,249.000000","19,384.720000",M2044282225,0.000000,0.000000,0,0
2,1,TRANSFER,181.000000,C1305486145,181.000000,0.000000,C553264065,0.000000,0.000000,1,0
3,1,CASH_OUT,181.000000,C840083671,181.000000,0.000000,C38997010,"21,182.000000",0.000000,1,0
4,1,PAYMENT,"11,668.140000",C2048537720,"41,554.000000","29,885.860000",M1230701703,0.000000,0.000000,0,0


## 3. Validação estrutural

Antes de transformar a base, verificamos se:

- todas as colunas esperadas estão presentes;
- a variável alvo `isFraud` possui apenas os valores 0 e 1;
- existem valores ausentes;
- existem duplicatas.

Como a EDA indicou ausência de nulos e duplicatas, esta etapa funciona principalmente como uma checagem de segurança.


In [20]:
colunas_faltantes = [col for col in COLUNAS_ESPERADAS if col not in df.columns]

if colunas_faltantes:
    raise ValueError(f"Colunas ausentes no dataset: {colunas_faltantes}")

valores_alvo = sorted(df["isFraud"].dropna().unique().tolist())

if valores_alvo != [0, 1]:
    raise ValueError(f"A variável isFraud deveria conter apenas 0 e 1. Valores encontrados: {valores_alvo}")

tipos_encontrados = sorted(df["type"].dropna().unique().tolist())
tipos_inesperados = sorted(set(tipos_encontrados) - set(TIPOS_TRANSACAO_ESPERADOS))

if tipos_inesperados:
    print(f"Atenção: tipos de transação inesperados encontrados: {tipos_inesperados}")

nulos_por_coluna = df.isna().sum()
total_nulos = int(nulos_por_coluna.sum())

duplicatas = int(df.duplicated().sum())

resumo_qualidade = pd.DataFrame({
    "item": [
        "linhas",
        "colunas",
        "total_nulos",
        "duplicatas",
        "tipos_transacao_encontrados",
    ],
    "valor": [
        df.shape[0],
        df.shape[1],
        total_nulos,
        duplicatas,
        ", ".join(tipos_encontrados),
    ],
})

resumo_qualidade

,item,valor
0,linhas,6362620
1,colunas,11
2,total_nulos,0
3,duplicatas,0
4,tipos_transacao_encontrados,"CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER"


In [21]:
# Remoção segura de duplicatas, caso existam.
# Na EDA, o esperado é que duplicatas = 0.
if duplicatas > 0:
    print(f"Removendo {duplicatas:,} duplicatas...")
    df = df.drop_duplicates().reset_index(drop=True)

# Se existissem nulos, o correto seria analisar caso a caso.
# Como a base PaySim normalmente não possui nulos, interrompemos para evitar tratamento automático indevido.
if total_nulos > 0:
    display(nulos_por_coluna[nulos_por_coluna > 0].to_frame("nulos"))
    raise ValueError(
        "Foram encontrados valores ausentes. "
        "Analise os nulos antes de seguir com o pré-processamento."
    )

print(f"Dimensão após validação: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas")

Dimensão após validação: 6,362,620 linhas x 11 colunas


## 4. Resumo do desbalanceamento original

Antes de qualquer transformação, registramos a proporção entre transações legítimas e fraudulentas.

Esse número será usado depois para comparar se as divisões de treino, validação e teste preservaram um cenário coerente de desbalanceamento.


In [22]:
def resumo_alvo(dados: pd.DataFrame, nome: str, coluna_alvo: str = "isFraud") -> dict:
    total = len(dados)
    fraudes = int(dados[coluna_alvo].sum())
    legitimas = int(total - fraudes)
    taxa_fraude = fraudes / total if total > 0 else 0
    imbalance_ratio = legitimas / fraudes if fraudes > 0 else np.inf

    return {
        "conjunto": nome,
        "total": total,
        "legitimas": legitimas,
        "fraudes": fraudes,
        "taxa_fraude": taxa_fraude,
        "imbalance_ratio_legitimas_por_fraude": imbalance_ratio,
    }


pd.DataFrame([resumo_alvo(df, "base_original")])

,conjunto,total,legitimas,fraudes,taxa_fraude,imbalance_ratio_legitimas_por_fraude
0,base_original,6362620,6354407,8213,0.001291,773.701084


## 5. Otimização inicial de tipos

O PaySim possui mais de 6 milhões de registros. Por isso, algumas conversões simples ajudam a reduzir o consumo de memória.

Essa etapa não altera o significado das variáveis.


In [23]:
def memoria_mb(dataframe: pd.DataFrame) -> float:
    return dataframe.memory_usage(deep=True).sum() / (1024 ** 2)


memoria_antes = memoria_mb(df)

colunas_float = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

for col in colunas_float:
    df[col] = pd.to_numeric(df[col], errors="raise").astype("float32")

df["step"] = pd.to_numeric(df["step"], errors="raise").astype("int32")
df["isFraud"] = df["isFraud"].astype("int8")
df["isFlaggedFraud"] = df["isFlaggedFraud"].astype("int8")
df["type"] = df["type"].astype("category")

memoria_depois = memoria_mb(df)

print(f"Memória antes: {memoria_antes:.2f} MB")
print(f"Memória depois: {memoria_depois:.2f} MB")
print(f"Redução: {(1 - memoria_depois / memoria_antes) * 100:.2f}%")

Memória antes: 1452.57 MB
Memória depois: 885.69 MB
Redução: 39.03%


## 6. Remoção de colunas que não entrarão no modelo principal

Com base na EDA, as seguintes colunas não serão usadas como features principais:

- `nameOrig`: identificador de origem com alta cardinalidade;
- `nameDest`: identificador de destino com alta cardinalidade;
- `isFlaggedFraud`: sinalização interna do dataset, com risco de vazamento e baixo recall.

A coluna `isFlaggedFraud` poderá ser usada depois apenas como baseline de comparação, mas não como entrada dos modelos supervisionados principais.


In [24]:
colunas_removidas_modelo = [
    "nameOrig",
    "nameDest",
    "isFlaggedFraud",
]

df_base = df.drop(columns=colunas_removidas_modelo).copy()

print("Colunas removidas do modelo principal:")
for col in colunas_removidas_modelo:
    print(f"- {col}")

print(f"Dimensão após remoção: {df_base.shape[0]:,} linhas x {df_base.shape[1]:,} colunas")
df_base.head()

Colunas removidas do modelo principal:
- nameOrig
- nameDest
- isFlaggedFraud
Dimensão após remoção: 6,362,620 linhas x 8 colunas


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,"9,839.639648","170,136.000000","160,296.359375",0.000000,0.000000,0
1,1,PAYMENT,"1,864.280029","21,249.000000","19,384.720703",0.000000,0.000000,0
2,1,TRANSFER,181.000000,181.000000,0.000000,0.000000,0.000000,1
3,1,CASH_OUT,181.000000,181.000000,0.000000,"21,182.000000",0.000000,1
4,1,PAYMENT,"11,668.139648","41,554.000000","29,885.859375",0.000000,0.000000,0


## 7. Engenharia de atributos

Nesta etapa criamos as features definidas a partir da EDA:

### Valor da transação

- `log_amount`: transformação logarítmica de `amount`, útil por causa da forte assimetria.

### Tempo

- `dia_simulado`: dia da simulação, considerando que cada `step` representa uma hora;
- `hora_simulada`: hora dentro do dia simulado.

### Saldos

- `erro_saldo_origem`: diferença entre saldo esperado e saldo final na origem;
- `erro_saldo_destino`: diferença entre saldo esperado e saldo final no destino;
- versões absolutas dos erros;
- flags de inconsistência de saldo.

### Tipo da transação

- codificação manual one-hot de `type`, usando categorias esperadas do PaySim.

A codificação manual evita depender de categorias aprendidas a partir do conjunto completo.


In [25]:
df_proc = pd.DataFrame(index=df_base.index)

# Variáveis temporais
df_proc["step"] = df_base["step"].astype("int32")
df_proc["dia_simulado"] = ((df_base["step"] - 1) // 24 + 1).astype("int16")
df_proc["hora_simulada"] = ((df_base["step"] - 1) % 24).astype("int8")

# Valor da transação
df_proc["amount"] = df_base["amount"].astype("float32")
df_proc["log_amount"] = np.log1p(df_base["amount"]).astype("float32")

# Saldos originais
df_proc["oldbalanceOrg"] = df_base["oldbalanceOrg"].astype("float32")
df_proc["newbalanceOrig"] = df_base["newbalanceOrig"].astype("float32")
df_proc["oldbalanceDest"] = df_base["oldbalanceDest"].astype("float32")
df_proc["newbalanceDest"] = df_base["newbalanceDest"].astype("float32")

# Erros de saldo
erro_saldo_origem = df_base["oldbalanceOrg"] - df_base["amount"] - df_base["newbalanceOrig"]
erro_saldo_destino = df_base["oldbalanceDest"] + df_base["amount"] - df_base["newbalanceDest"]

df_proc["erro_saldo_origem"] = erro_saldo_origem.astype("float32")
df_proc["erro_saldo_destino"] = erro_saldo_destino.astype("float32")
df_proc["erro_saldo_origem_abs"] = erro_saldo_origem.abs().astype("float32")
df_proc["erro_saldo_destino_abs"] = erro_saldo_destino.abs().astype("float32")

df_proc["inconsistencia_saldo_origem"] = (erro_saldo_origem.abs() > TOLERANCIA_SALDO).astype("int8")
df_proc["inconsistencia_saldo_destino"] = (erro_saldo_destino.abs() > TOLERANCIA_SALDO).astype("int8")

# Flags simples de saldo zero
df_proc["origem_saldo_inicial_zero"] = (df_base["oldbalanceOrg"] == 0).astype("int8")
df_proc["origem_saldo_final_zero"] = (df_base["newbalanceOrig"] == 0).astype("int8")
df_proc["destino_saldo_inicial_zero"] = (df_base["oldbalanceDest"] == 0).astype("int8")
df_proc["destino_saldo_final_zero"] = (df_base["newbalanceDest"] == 0).astype("int8")

# One-hot encoding manual da variável type
for tipo in TIPOS_TRANSACAO_ESPERADOS:
    df_proc[f"type_{tipo}"] = (df_base["type"] == tipo).astype("int8")

# Variável alvo
df_proc["isFraud"] = df_base["isFraud"].astype("int8")

print(f"Dimensão da base processada: {df_proc.shape[0]:,} linhas x {df_proc.shape[1]:,} colunas")
df_proc.head()

Dimensão da base processada: 6,362,620 linhas x 25 colunas


,step,dia_simulado,hora_simulada,amount,log_amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,erro_saldo_origem,erro_saldo_destino,erro_saldo_origem_abs,erro_saldo_destino_abs,inconsistencia_saldo_origem,inconsistencia_saldo_destino,origem_saldo_inicial_zero,origem_saldo_final_zero,destino_saldo_inicial_zero,destino_saldo_final_zero,type_PAYMENT,type_TRANSFER,type_CASH_OUT,type_DEBIT,type_CASH_IN,isFraud
0,1,1,0,"9,839.639648",9.194276,"170,136.000000","160,296.359375",0.000000,0.000000,0.000000,"9,839.639648",0.000000,"9,839.639648",0,1,0,0,1,1,1,0,0,0,0,0
1,1,1,0,"1,864.280029",7.531167,"21,249.000000","19,384.720703",0.000000,0.000000,0.000000,"1,864.280029",0.000000,"1,864.280029",0,1,0,0,1,1,1,0,0,0,0,0
2,1,1,0,181.000000,5.204007,181.000000,0.000000,0.000000,0.000000,0.000000,181.000000,0.000000,181.000000,0,1,0,1,1,1,0,1,0,0,0,1
3,1,1,0,181.000000,5.204007,181.000000,0.000000,"21,182.000000",0.000000,0.000000,"21,363.000000",0.000000,"21,363.000000",0,1,0,1,0,1,0,0,1,0,0,1
4,1,1,0,"11,668.139648",9.364703,"41,554.000000","29,885.859375",0.000000,0.000000,0.000000,"11,668.139648",0.000000,"11,668.139648",0,1,0,0,1,1,1,0,0,0,0,0


## 8. Conferência das features criadas

Nesta etapa conferimos:

- se não existem valores ausentes após a engenharia de atributos;
- se todas as colunas estão em formato numérico;
- quais colunas serão usadas como features;
- qual coluna será usada como alvo.


In [26]:
total_nulos_processado = int(df_proc.isna().sum().sum())
colunas_nao_numericas = df_proc.select_dtypes(exclude=["number"]).columns.tolist()

if total_nulos_processado > 0:
    display(df_proc.isna().sum()[df_proc.isna().sum() > 0].to_frame("nulos"))
    raise ValueError("Foram gerados valores ausentes no pré-processamento.")

if colunas_nao_numericas:
    raise ValueError(f"Colunas não numéricas encontradas: {colunas_nao_numericas}")

COLUNA_ALVO = "isFraud"
FEATURES = [col for col in df_proc.columns if col != COLUNA_ALVO]

print(f"Quantidade de features: {len(FEATURES)}")
print(f"Coluna alvo: {COLUNA_ALVO}")

pd.DataFrame({
    "feature": FEATURES,
    "dtype": [str(df_proc[col].dtype) for col in FEATURES],
}).head(30)

Quantidade de features: 24
Coluna alvo: isFraud


,feature,dtype
0,step,int32
1,dia_simulado,int16
2,hora_simulada,int8
3,amount,float32
4,log_amount,float32
5,oldbalanceOrg,float32
6,newbalanceOrig,float32
7,oldbalanceDest,float32
8,newbalanceDest,float32
9,erro_saldo_origem,float32


## 9. Separação temporal em treino, validação e teste

A proposta do projeto recomenda uma separação temporal usando `step`, pois ela simula melhor um cenário real:

- o modelo aprende com transações mais antigas;
- valida em transações intermediárias;
- testa em transações futuras.

A divisão usada será:

- **70% dos steps iniciais:** treino;
- **15% dos steps seguintes:** validação;
- **15% dos steps finais:** teste.

Essa separação evita misturar transações futuras no treino.


In [27]:
def dividir_temporal_por_step(
    dados: pd.DataFrame,
    coluna_step: str = "step",
    proporcao_treino: float = 0.70,
    proporcao_validacao: float = 0.15,
):
    if not 0 < proporcao_treino < 1:
        raise ValueError("proporcao_treino deve estar entre 0 e 1.")

    if not 0 < proporcao_validacao < 1:
        raise ValueError("proporcao_validacao deve estar entre 0 e 1.")

    if proporcao_treino + proporcao_validacao >= 1:
        raise ValueError("A soma de treino e validação deve ser menor que 1.")

    steps_unicos = np.sort(dados[coluna_step].unique())

    indice_fim_treino = int(len(steps_unicos) * proporcao_treino)
    indice_fim_validacao = int(len(steps_unicos) * (proporcao_treino + proporcao_validacao))

    step_fim_treino = steps_unicos[indice_fim_treino - 1]
    step_fim_validacao = steps_unicos[indice_fim_validacao - 1]

    treino = dados[dados[coluna_step] <= step_fim_treino].copy()
    validacao = dados[
        (dados[coluna_step] > step_fim_treino) &
        (dados[coluna_step] <= step_fim_validacao)
    ].copy()
    teste = dados[dados[coluna_step] > step_fim_validacao].copy()

    info = {
        "step_min": int(steps_unicos.min()),
        "step_max": int(steps_unicos.max()),
        "total_steps": int(len(steps_unicos)),
        "step_fim_treino": int(step_fim_treino),
        "step_fim_validacao": int(step_fim_validacao),
        "step_inicio_teste": int(step_fim_validacao + 1),
    }

    return treino, validacao, teste, info


treino, validacao, teste, info_divisao = dividir_temporal_por_step(df_proc)

print("Informações da divisão temporal:")
for chave, valor in info_divisao.items():
    print(f"{chave}: {valor}")

print()
print(f"Treino: {treino.shape}")
print(f"Validação: {validacao.shape}")
print(f"Teste: {teste.shape}")

Informações da divisão temporal:
step_min: 1
step_max: 743
total_steps: 743
step_fim_treino: 520
step_fim_validacao: 631
step_inicio_teste: 632

Treino: (6082007, 25)
Validação: (191147, 25)
Teste: (89466, 25)


## 10. Avaliação do desbalanceamento em cada divisão

Depois da separação temporal, verificamos se os conjuntos possuem fraudes suficientes para modelagem e avaliação.

Caso algum conjunto fique sem fraudes, a estratégia de divisão precisará ser revista.


In [28]:
resumo_divisoes = pd.DataFrame([
    resumo_alvo(treino, "treino"),
    resumo_alvo(validacao, "validacao"),
    resumo_alvo(teste, "teste"),
])

resumo_divisoes

,conjunto,total,legitimas,fraudes,taxa_fraude,imbalance_ratio_legitimas_por_fraude
0,treino,6082007,6076226,5781,0.000951,"1,051.068327"
1,validacao,191147,189967,1180,0.006173,160.988983
2,teste,89466,88214,1252,0.013994,70.458466


In [29]:
conjuntos_sem_fraude = resumo_divisoes.loc[resumo_divisoes["fraudes"] == 0, "conjunto"].tolist()

if conjuntos_sem_fraude:
    raise ValueError(
        "A divisão temporal gerou conjunto(s) sem fraude: "
        f"{conjuntos_sem_fraude}. "
        "Revise a estratégia de divisão antes de seguir."
    )

print("Todos os conjuntos possuem registros de fraude.")

Todos os conjuntos possuem registros de fraude.


### Observação sobre a divisão temporal

A separação temporal preserva a ordem dos eventos e simula um cenário mais próximo de uso real, em que o modelo é treinado com transações anteriores e avaliado em transações futuras.

Entretanto, a distribuição das fraudes não é uniforme ao longo do tempo. Por isso, os conjuntos de validação e teste ficaram com taxa de fraude maior do que a base original. Isso indica uma mudança temporal na distribuição da variável alvo.

Para complementar essa avaliação, o notebook de modelagem deverá incluir testes de robustez com diferentes imbalance ratios, como 1:100, 1:500 e 1:1000, conforme definido na proposta do projeto.

## 11. Conferência de colunas finais

Os três conjuntos devem possuir exatamente as mesmas colunas, na mesma ordem.


In [30]:
assert list(treino.columns) == list(validacao.columns) == list(teste.columns)

print("Colunas finais conferidas com sucesso.")
print(f"Total de colunas: {len(treino.columns)}")

pd.DataFrame({
    "coluna": treino.columns,
    "dtype_treino": [str(treino[col].dtype) for col in treino.columns],
    "dtype_validacao": [str(validacao[col].dtype) for col in validacao.columns],
    "dtype_teste": [str(teste[col].dtype) for col in teste.columns],
})

Colunas finais conferidas com sucesso.
Total de colunas: 25


,coluna,dtype_treino,dtype_validacao,dtype_teste
0,step,int32,int32,int32
1,dia_simulado,int16,int16,int16
2,hora_simulada,int8,int8,int8
3,amount,float32,float32,float32
4,log_amount,float32,float32,float32
5,oldbalanceOrg,float32,float32,float32
6,newbalanceOrig,float32,float32,float32
7,oldbalanceDest,float32,float32,float32
8,newbalanceDest,float32,float32,float32
9,erro_saldo_origem,float32,float32,float32


## 12. Salvamento dos arquivos processados

Serão salvos:

- `treino_processado.parquet`;
- `validacao_processado.parquet`;
- `teste_processado.parquet`;
- `metadata_preprocessamento.json`.

Caso o ambiente não tenha suporte a Parquet, o notebook salva automaticamente em CSV.


In [31]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def salvar_dataframe(dados: pd.DataFrame, nome: str) -> str:
    caminho_parquet = OUTPUT_DIR / f"{nome}.parquet"
    caminho_csv = OUTPUT_DIR / f"{nome}.csv"

    try:
        dados.to_parquet(caminho_parquet, index=False)
        return str(caminho_parquet)
    except Exception as erro:
        print(f"Não foi possível salvar {nome} em Parquet.")
        print(f"Motivo: {erro}")
        print(f"Salvando {nome} em CSV...")
        dados.to_csv(caminho_csv, index=False)
        return str(caminho_csv)


arquivos_salvos = {
    "treino": salvar_dataframe(treino, "treino_processado"),
    "validacao": salvar_dataframe(validacao, "validacao_processado"),
    "teste": salvar_dataframe(teste, "teste_processado"),
}

arquivos_salvos

{'treino': 'data/processado/treino_processado.parquet',
 'validacao': 'data/processado/validacao_processado.parquet',
 'teste': 'data/processado/teste_processado.parquet'}

In [32]:
metadata = {
    "data_execucao": datetime.now().isoformat(timespec="seconds"),
    "dataset_origem": str(dataset_path),
    "output_dir": str(OUTPUT_DIR),
    "coluna_alvo": COLUNA_ALVO,
    "features": FEATURES,
    "colunas_removidas_modelo": colunas_removidas_modelo,
    "tipos_transacao_esperados": TIPOS_TRANSACAO_ESPERADOS,
    "tolerancia_saldo": TOLERANCIA_SALDO,
    "info_divisao_temporal": info_divisao,
    "resumo_divisoes": resumo_divisoes.to_dict(orient="records"),
    "arquivos_salvos": arquivos_salvos,
    "observacoes": [
        "SMOTE não foi aplicado neste notebook.",
        "Técnicas de balanceamento devem ser aplicadas somente ao conjunto de treino no notebook de modelagem.",
        "isFlaggedFraud foi removida das features principais para evitar vazamento de informação.",
        "nameOrig e nameDest foram removidas por alta cardinalidade.",
    ],
}

caminho_metadata = OUTPUT_DIR / "metadata_preprocessamento.json"

with open(caminho_metadata, "w", encoding="utf-8") as arquivo:
    json.dump(metadata, arquivo, ensure_ascii=False, indent=4)

print(f"Metadata salvo em: {caminho_metadata}")

Metadata salvo em: data/processado/metadata_preprocessamento.json


## 13. Como usar estes arquivos no notebook de modelagem

No próximo notebook, a modelagem deve começar carregando os arquivos processados:

```python
import pandas as pd

treino = pd.read_parquet("data/processado/treino_processado.parquet")
validacao = pd.read_parquet("data/processado/validacao_processado.parquet")
teste = pd.read_parquet("data/processado/teste_processado.parquet")

X_train = treino.drop(columns=["isFraud"])
y_train = treino["isFraud"]

X_valid = validacao.drop(columns=["isFraud"])
y_valid = validacao["isFraud"]

X_test = teste.drop(columns=["isFraud"])
y_test = teste["isFraud"]
```

A partir disso, o notebook de modelagem poderá comparar:

- baseline tudo legítimo;
- baseline por regra;
- Regressão Logística;
- Random Forest;
- XGBoost;
- estratégias de desbalanceamento;
- ajuste de threshold;
- tempo de inferência;
- matriz de custo.
